# 0. Drive Bağlantısı ve Dosya Yapısını Oluşturma


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/slam_asr")
DATA_ROOT = PROJECT_ROOT / "data" / "LibriSpeech"
ARCHIVE_DIR = PROJECT_ROOT / "data" / "archives"
MANIFEST_DIR = PROJECT_ROOT / "manifests"
RUNS_DIR = PROJECT_ROOT / "runs"

for directory in (DATA_ROOT, ARCHIVE_DIR, MANIFEST_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Veri dizini:", DATA_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Veri dizini: /content/drive/MyDrive/slam_asr/data/LibriSpeech


# 1. Veri Seti ve Çalışma Gereksinimlerinin Hazırlanması

## 1.1 Veri Setini İndirme

In [ ]:
!bash /content/drive/MyDrive/slam_asr/01_download_librispeech.sh

Filesystem      Size  Used Avail Use% Mounted on
drive           226G   12G  215G   6% /content/drive
İndiriliyor: train-clean-100.tar.gz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 6091M  100 6091M    0     0  1758k      0  0:59:07  0:59:07 --:--:-- 2740k
Doğrulandı: train-clean-100.tar.gz
İndiriliyor: train-clean-360.tar.gz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 21.4G  100 21.4G    0     0  1923k      0  3:15:04  3:15:04 --:--:-- 5304k
Doğrulandı: train-clean-360.tar.gz
İndiriliyor: train-other-500.tar.gz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 28.4G  100 28.4G    0     0  1635k      0  5:04:23  5:04:23 --:--:-- 2317k
Doğruland

## 1.2 Tranining clean-100 verisini referans metinle eşleştirme

In [ ]:
!python -u /content/drive/MyDrive/slam_asr/02_prepare_librispeech.py

28539 pairs, 100.59 hours
/content/drive/MyDrive/slam_asr/manifests/train-clean-100.jsonl
/content/drive/MyDrive/slam_asr/manifests/pilot_64_seed42.jsonl


In [ ]:
import json
from pathlib import Path
from IPython.display import Audio, display

root = Path("/content/drive/MyDrive/slam_asr")
manifest = root / "manifests" / "pilot_64_seed42.jsonl"
rows = [json.loads(line) for line in manifest.read_text().splitlines()]

assert len(rows) == 64, f"64 kayıt bekleniyordu, {len(rows)} bulundu."
assert len({row["id"] for row in rows}) == 64, "Tekrarlanan kayıt kimliği var."

sample = rows[0]
print(f"Pilot: {len(rows)} kayıt, toplam {sum(r['duration_seconds'] for r in rows)/60:.1f} dakika")
print(f"Örnek: {sample['id']} · {sample['duration_seconds']:.1f} saniye")
print("Referans:", sample["text"])
display(Audio(filename=str(root / sample["audio_path"])))

Pilot: 64 kayıt, toplam 12.8 dakika
Örnek: 6529-62554-0056 · 12.1 saniye
Referans: AND NOW SIX CONVICTS OF THE WORST DESCRIPTION PERHAPS WERE ROAMING OVER THEIR ISLAND IT WAS SERIOUS CERTAINLY AND TO LESS BRAVE MEN IT WOULD HAVE BEEN SECURITY LOST NO MATTER


## 1.2 Ortam Gerekliliklerinin Yüklenmesi

In [ ]:
%pip -q install -r requirements.txt

## 1.3 LibriSpeech-960 eğitim manifestinin hazırlanması

Bu adım, eğitim kayıtlarını referans transkriptleriyle eşleştirerek
`manifests/train-960.jsonl` dosyasını oluşturur.

In [ ]:
!python -u /content/drive/MyDrive/slam_asr/06_prepare_train960.py --progress-every 10000

Taranıyor: train-clean-100
train-clean-100: 10,000 ses tarandı
train-clean-100: 20,000 ses tarandı
Taranıyor: train-clean-360
train-clean-360: 10,000 ses tarandı
train-clean-360: 20,000 ses tarandı
train-clean-360: 30,000 ses tarandı
train-clean-360: 40,000 ses tarandı
train-clean-360: 50,000 ses tarandı
train-clean-360: 60,000 ses tarandı
train-clean-360: 70,000 ses tarandı
train-clean-360: 80,000 ses tarandı
train-clean-360: 90,000 ses tarandı
train-clean-360: 100,000 ses tarandı
Taranıyor: train-other-500
train-other-500: 10,000 ses tarandı
train-other-500: 20,000 ses tarandı
train-other-500: 30,000 ses tarandı
train-other-500: 40,000 ses tarandı
train-other-500: 50,000 ses tarandı
train-other-500: 60,000 ses tarandı
train-other-500: 70,000 ses tarandı
train-other-500: 80,000 ses tarandı
train-other-500: 90,000 ses tarandı
train-other-500: 100,000 ses tarandı
train-other-500: 110,000 ses tarandı
train-other-500: 120,000 ses tarandı
train-other-500: 130,000 ses tarandı
train-other-50

# 2. Deneyler

## 2.1 Deney A — Whisper-small ve Vicuna-7B ile projektör eğitimi

LibriSpeech-960 eğitim verisi kullanılır. Whisper-small ses kodlayıcısı
ve Vicuna-7B dil modeli sabit tutulur. Yalnızca projektör eğitilir.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 07_train_vicuna_full.py \
    --run-name A_vicuna_train960 \
    --batch-size 4 \
    --gradient-accumulation-steps 1 \
    --lr 1e-4 \
    --warmup-steps 1000 \
    --weight-decay 0 \
    --max-steps 100000 \
    --save-every 500 \
    --log-every 25 \
    --eval-every 2000 \
    --dev-samples 64 \
    --patience 5 \
    --seed 42

/content/drive/MyDrive/slam_asr
Koşu: /content/drive/MyDrive/slam_asr/runs/A_vicuna_train960
config.json: 100% 615/615 [00:00<00:00, 4.94MB/s]
Loading weights: 100% 479/479 [00:00<00:00, 17387.48it/s]
tokenizer_config.json: 100% 749/749 [00:00<00:00, 6.57MB/s]

tokenizer.model: downloading bytes:   0% 0.00/500k [00:00<?, ?B/s]
tokenizer.model: downloading bytes: 100% 346k/346k [00:00<00:00, 1.24MB/s, 34.4kB/s  ]
tokenizer.model: reconstructing file: 100% 500k/500k [00:00<00:00, 1.79MB/s, 49.7kB/s  ]
special_tokens_map.json: 100% 438/438 [00:00<00:00, 4.89MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin.index.json: 100% 26.8k/26.8k [00:00<00:00, 138MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/9.98G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/13.5G [00:00<?, ?B/s]


model.safetensors.index

## 2.2 Deney B — Qwen3.5-0.8B ile projektör eğitimi

Whisper-small ses kodlayıcısı ve Qwen3.5-0.8B Base dil modeli
dondurulur. LibriSpeech-960 eğitim havuzu kullanılarak yalnızca
ReLU içeren MLP projektör eğitilir.

Her 2.000 adımda `dev-other` kümesinden seçilen 64 kayıt üzerinde
WER ölçülür. En düşük doğrulama WER’ine sahip projektör `best`
olarak kaydedilir ve GRPO deneylerinde başlangıç olarak kullanılır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 08_qwen35_asr.py \
    --mode train \
    --run-name B_qwen08_train960 \
    --batch-size 4 \
    --gradient-accumulation-steps 1 \
    --lr 1e-4 \
    --warmup-steps 1000 \
    --weight-decay 0 \
    --max-steps 100000 \
    --eval-every 2000 \
    --dev-samples 64 \
    --patience 5 \
    --save-every 500 \
    --log-every 25 \
    --max-new-tokens 160 \
    --seed 42

/content/drive/MyDrive/slam_asr
Koşu: /content/drive/MyDrive/slam_asr/runs/B_qwen08_train960
Loading weights: 100% 479/479 [00:00<00:00, 17234.60it/s]
Loading weights: 100% 320/320 [00:00<00:00, 16389.20it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
{'step': 1, 'completed_epochs': 0, 'next_offset': 4, 'batch_loss': 4.259253978729248, 'lr': 1.0000000000000001e-07, 'tokens': 194, 'peak_gpu_gib': 12.539956092834473, 'unix_time': 1794482152.030457}
{'step': 25, 'completed_epochs': 0, 'next_offset': 100, 'batch_loss': 4.114112377166748, 'lr': 2.5e-06, 'tokens': 250, 'peak_gpu_gib': 12.

## 2.3 A ve B Modellerinin Doğrulama ve Testleri

### 2.3.1 Deney A ve B — Tam dev-other değerlendirmesi

A ve B deneylerinin `best` ve `last` kontrol noktaları, tam
`dev-other` kümesinde değerlendirilir. Transkriptler greedy decoding
(`beam=1`) ile üretilir. WER ve referans transkript üzerindeki
token düzeyinde kayıp raporlanır.

Çıktı klasörü: `AB_full_dev_greedy`

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 10_evaluate_asr.py \
    --runs A_vicuna_train960 B_qwen08_train960 \
    --checkpoints best last \
    --splits dev-other \
    --beams 1 \
    --max-new-tokens 160 \
    --loss \
    --output AB_full_dev_greedy

Görüntülenen çıkış son 5000 satıra kısaltıldı.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum

### 2.3.2 Deney A ve B — Beam search ile test değerlendirmesi

A ve B deneylerinin `best` ve `last` kontrol noktaları, `test-clean`
ve `test-other` kümelerinde dört genişlikli beam search (`beam=4`)
ile değerlendirilir. Her test kümesi için WER ayrı hesaplanır.

Çıktı klasörü: `AB_full_test_beam4`

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 10_evaluate_asr.py \
    --runs A_vicuna_train960 B_qwen08_train960 \
    --checkpoints best last \
    --splits test-clean test-other \
    --beams 4 \
    --max-new-tokens 160 \
    --output AB_full_test_beam4

Görüntülenen çıkış son 5000 satıra kısaltıldı.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum

# 3. GRPO Eğitimleri

## 3.1 GRPO öncesi ortam kontrolü

Eğitim betiklerinin kullandığı bileşenler içe aktarılır ve paket
sürümleri yazdırılır. Seçilen GPU’nun CUDA ve BF16 desteği kontrol edilir.
Bu hücre eğitim başlatmaz.

In [ ]:
from importlib.metadata import version

import torch
import soundfile
from huggingface_hub import hf_hub_download
from peft import (
    LoraConfig, get_peft_model,
    get_peft_model_state_dict, set_peft_model_state_dict,
)
from transformers import (
    AutoTokenizer, GenerationConfig, Qwen3_5ForCausalLM,
    WhisperFeatureExtractor, WhisperModel,
)

for package in ("torch", "transformers", "peft", "huggingface_hub", "soundfile"):
    print(f"{package}: {version(package)}")

assert torch.cuda.is_available(), "Colab çalışma zamanında GPU seçilmeli."
assert torch.cuda.is_bf16_supported(), "Bu betik BF16 destekli GPU bekliyor."
print("PASS — C betiğinin importları ve BF16 GPU kontrolü başarılı.")
print("GPU:", torch.cuda.get_device_name(0))

torch: 2.14.1+cu130
transformers: 5.22.1
peft: 0.21.2
huggingface_hub: 1.33.0
soundfile: 0.14.0
PASS — C betiğinin importları ve BF16 GPU kontrolü başarılı.
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 3.2 Deney C — Negatif WER ödülü ile GRPO

Eğitim B-best projektöründen başlar. Her güncellemede 32 ses kaydı
kullanılır ve kayıt başına 16 aday transkript üretilir. Adayların ödülü,
referansa göre hesaplanan kelime hata oranının negatifidir.
Koşunun eğitim bütçesi 800 adımdır.

In [ ]:
from pathlib import Path
import runpy
import subprocess
import sys

root = Path("/content/drive/MyDrive/slam_asr")
script = root / "09_grpo_qwen_wer.py"

# Eski betikle yanlışlıkla ham EM deneyini başlatmayı önler.
assert runpy.run_path(str(script))["VERSION"] == "2.2.0", "Drive'daki betiği güncelle."

experiments = [
    ("wer", "C_qwen08_lora_negwer"),
    ("em",  "C_qwen08_lora_em_strip"),
    ("ed",  "C_qwen08_lora_ed"),
]

for reward, run_name in experiments:
    run_dir = root / "runs" / run_name
    command = [
        sys.executable, "-u", str(script),
        "--root", str(root),
        "--b-run", "B_qwen08_train960",
        "--reward", reward,
        "--run-name", run_name,
        "--steps", "1000",
    ]

    # Hücre yeniden çalıştırılırsa betik kayıtlı adım ve ayarları kontrol eder.
    # 1000 adıma ulaşmış koşuda yeni eğitim adımı atılmaz.
    if (run_dir / "last.pt").is_file():
        command.append("--resume")

    print(f"\n>>> Ödül: {reward.upper()} | Koşu: {run_name}", flush=True)
with subprocess.Popen(
        command,
        cwd=root,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
) as process:
        for line in process.stdout:
            print(line, end="", flush=True)

        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)


>>> Ödül: WER | Koşu: C_qwen08_lora_negwer

>>> Ödül: EM | Koşu: C_qwen08_lora_em_strip

>>> Ödül: ED | Koşu: C_qwen08_lora_ed

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 17405.41it/s]

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 16394.61it/s]
Eğitilebilir Qwen-LoRA parametresi: 638976
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers]

## 3.2.1 Deney C — Negatif WER ile eğitilen modelin değerlendirilmesi

C deneyinin `best` ve `last` kontrol noktaları tam `dev-other`,
`test-clean` ve `test-other` kümelerinde greedy decoding (`beam=1`)
ile değerlendirilir. `best`, eğitim sırasında doğrulama WER’ine
göre seçilmiş kontrol noktasıdır. Sonuçlar `C_full_eval` klasörüne yazılır.

In [ ]:
!python -u 10_evaluate_asr.py \
    --runs C_qwen08_wer_32x16_s800 \
    --checkpoints best last \
    --splits dev-other test-clean test-other \
    --beams 1 \
    --max-new-tokens 160 \
    --output C_full_eval

## 3.3 Deney D — Tam eşleşme ve parçalı WER ödülüyle GRPO

Eğitim B-best projektöründen başlar. Her güncellemede 32 ses kaydı
kullanılır ve her kayıt için 16 aday transkript üretilir.
Ödül, büyük/küçük harf ayrımı gözetmeyen tam eşleşme ile
eşik temelli WER puanını birleştirir.

F bonus deneyleriyle karşılaştırmada D’nin 810. adımındaki
kontrol noktası kullanılır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 11_grpo_qwen_custom.py \
    --b-run B_qwen08_train960 \
    --run-name D_qwen08_em40_wer60 \
    --em-weight 0.4 \
    --wer-threshold 0.10 \
    --excess-penalty 2 \
    --sounds-per-step 2 \
    --group-size 4 \
    --lr 1e-6 \
    --beta 0.04 \
    --seed 42 \
    --steps 1000

/content/drive/MyDrive/slam_asr
train-clean-100.tar.gz: 28539 ses yerel diske hazırlanıyor.
train-clean-360.tar.gz: 104014 ses yerel diske hazırlanıyor.
train-other-500.tar.gz: 148688 ses yerel diske hazırlanıyor.
preprocessor_config.json: 100% 185k/185k [00:00<00:00, 518MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 12.5MB/s]

model.safetensors: downloading bytes:  10% 95.5M/967M [00:00<00:06, 140MB/s, 4.55MB/s  ] 
model.safetensors: downloading bytes:  15% 143M/967M [00:01<00:03, 208MB/s, 9.28MB/s  ] 
model.safetensors: reconstructing file:  19% 180M/967M [00:01<00:04, 186MB/s, 6.52MB/s  ]
model.safetensors: downloading bytes:  25% 238M/967M [00:01<00:03, 226MB/s, 20.2MB/s  ]
model.safetensors: downloading bytes:  30% 289M/967M [00:01<00:02, 235MB/s, 24.8MB/s  ]
model.safetensors: downloading bytes:  35% 337M/967M [00:01<00:02, 236MB/s, 28.9MB/s  ]
model.safetensors: downloading bytes:  39% 375M/967M [00:01<00:02, 263MB/s, 30.8MB/s  ]
model.safetensors: reconstructing file:  62% 5

## 3.4 Deney E — Qwen3.5-9B ile projektör eğitimi

Whisper-small ses kodlayıcısı ve Qwen3.5-9B Base dil modeli
sabit tutulur. LibriSpeech-960 eğitim havuzu kullanılarak
yalnızca ReLU içeren MLP projektör eğitilir.

Eğitim sırasında her 2.000 adımda `dev-other` kümesinden
seçilen 64 kayıt üzerinde WER ölçülür. `100.000` adım,
eğitim için belirlenen üst sınırdır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 08_qwen35_asr.py \
    --mode train \
    --qwen Qwen/Qwen3.5-9B-Base \
    --run-name E_qwen9b_train960 \
    --batch-size 4 \
    --gradient-accumulation-steps 1 \
    --lr 1e-4 \
    --warmup-steps 1000 \
    --weight-decay 0 \
    --max-steps 100000 \
    --save-every 500 \
    --log-every 25 \
    --eval-every 2000 \
    --dev-samples 64 \
    --patience 5 \
    --num-beams 1 \
    --max-new-tokens 160 \
    --seed 42

/content/drive/MyDrive/slam_asr
Koşu: /content/drive/MyDrive/slam_asr/runs/E_qwen9b_train960
config.json: 100% 3.13k/3.13k [00:00<00:00, 18.8MB/s]
Loading weights: 100% 479/479 [00:00<00:00, 18361.94it/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 79.0MB/s]
model.safetensors.index.json: 100% 79.7k/79.7k [00:00<00:00, 362MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/8.69G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/14.0G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/14.0G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  25% 4.82G/19.3G [00:22<00:21, 660MB/s,  263MB/s  ]
Reconstructing (incomplete total...):  68% 13.1G/19.3G [00:47<00:19, 326MB/s,  264MB/s  ]
Reconstructing (incomplete total...):  68% 13.1G/19.3G [00:47<00:19, 326MB/s,  326MB/s  ]
Reconstructing (incomplete total...):  68% 13.1G

### 3.4.1 Deney E — Tam dev-other değerlendirmesi

`best` ve `last` kontrol noktaları tam `dev-other` kümesinde
greedy decoding (`beam=1`) ile değerlendirilir. Tam doğrulama
kümesinde daha düşük WER veren `last` kontrol noktası,
rapordaki test değerlendirmesi için seçilmiştir.

Çıktı klasörü: `E_qwen9b_full_dev`

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 10_evaluate_asr.py \
    --runs E_qwen9b_train960 \
    --checkpoints best last \
    --splits dev-other \
    --beams 1 \
    --max-new-tokens 160 \
    --output E_qwen9b_full_dev

/content/drive/MyDrive/slam_asr
preprocessor_config.json: 100% 185k/185k [00:00<00:00, 569MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 16.5MB/s]

model.safetensors: downloading bytes:   9% 91.7M/967M [00:01<00:05, 159MB/s, 6.52MB/s  ]
model.safetensors: downloading bytes:  12% 112M/967M [00:01<00:05, 171MB/s, 8.85MB/s  ] 
model.safetensors: downloading bytes:  22% 217M/967M [00:02<00:06, 120MB/s, 18.0MB/s  ]
model.safetensors: downloading bytes:  49% 469M/967M [00:03<00:01, 285MB/s, 35.9MB/s  ]
model.safetensors: downloading bytes:  54% 524M/967M [00:03<00:01, 358MB/s, 39.4MB/s  ]
model.safetensors: downloading bytes:  62% 595M/967M [00:03<00:01, 358MB/s, 46.8MB/s  ]
model.safetensors: downloading bytes:  66% 642M/967M [00:04<00:01, 309MB/s, 52.3MB/s  ]
model.safetensors: downloading bytes: 100% 643M/643M [00:04<00:00, 148MB/s, 52.7MB/s  ]
model.safetensors: reconstructing file: 100% 967M/967M [00:04<00:00, 222MB/s, 84.0MB/s  ]
Loading weights: 100% 479/479 [00:00<00:00, 17978.91i

## 3.5 Deney C — Negatif WER ödülüyle GRPO

Eğitim, B deneyinin `best` projektörüyle başlar. Ses kodlayıcısı
ve projektör sabit tutulur, Qwen3.5-0.8B modeline eklenen
LoRA parametreleri güncellenir.

Her adımda 32 ses kaydı kullanılır ve kayıt başına 16 aday
transkript üretilir. Her adayın ödülü, referans transkripte
göre hesaplanan WER’in negatifidir. Koşu 800 adım sürer.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

# C, B-best ağırlıklarından başlar. Her ses için 16 aday üretir.
# Her adayın ödülü = -(değiştirme + silme + ekleme) / referanstaki kelime sayısı.
!python -u 09_grpo_qwen_wer.py \
    --b-run B_qwen08_train960 \
    --run-name C_qwen08_wer_32x16_s800 \
    --reward wer \
    --steps 800 \
    --sounds-per-step 32 \
    --group-size 16 \
    --lr 1e-6 \
    --beta 0.04 \
    --clip 0.2 \
    --max-new-tokens 160 \
    --eval-every 200 \
    --save-every 50 \
    --seed 42

/content/drive/MyDrive/slam_asr
Loading weights: 100% 479/479 [00:00<00:00, 17469.88it/s]
Loading weights: 100% 320/320 [00:00<00:00, 16652.74it/s]
Eğitilebilir Qwen-LoRA parametresi: 638976
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attentio

### 3.5.1 Deney C — Doğrulama ve test değerlendirmesi

Negatif WER ödülüyle eğitilen C deneyinin `best` ve `last`
kontrol noktaları tam `dev-other`, `test-clean` ve `test-other`
kümelerinde greedy decoding (`beam=1`) ile değerlendirilir.
Çıktılar `C_qwen08_wer_s800_full_eval` klasörüne yazılır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 10_evaluate_asr.py \
    --runs C_qwen08_wer_32x16_s800 \
    --checkpoints best last \
    --splits dev-other test-clean test-other \
    --beams 1 \
    --max-new-tokens 160 \
    --output C_qwen08_wer_s800_full_eval

/content/drive/MyDrive/slam_asr
Loading weights: 100% 479/479 [00:00<00:00, 20579.27it/s]
Loading weights: 100% 320/320 [00:00<00:00, 19029.34it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but mu

### 3.5.2 B-best, C-best ve D-best — Kelime hatalarının karşılaştırılması

`test-clean` ve `test-other` tahminleri kayıt kimliğine göre
eşleştirilir ve referansların aynı olduğu doğrulanır. Her model
için ikame, silme ve ekleme sayıları ile toplam WER hesaplanır.
Model çiftleri için hata sayısı azalan, artan veya değişmeyen
kayıtlar ve net hata farkı raporlanır.

In [ ]:
from pathlib import Path
import pandas as pd

runs = Path("/content/drive/MyDrive/slam_asr/runs")
modeller = {
    "B": "B_qwen08_train960",
    "C": "C_qwen08_wer_32x16_s800",
    "D": "D_qwen08_em40_wer60_casefold_s32_g16_train960",
}

def tahminleri_oku(model, split):
    desen = f"*/{modeller[model]}/best/{split}/predictions.jsonl"
    dosyalar = sorted(runs.glob(desen))
    if not dosyalar:
        raise FileNotFoundError(f"Bulunamadı: {runs / desen}")

    # Aynı koşu birden fazla klasörde değerlendirilmişse çıktıları karşılaştır.
    alanlar = ["reference", "hypothesis", "S", "D", "I", "N"]
    tablolar = []
    for dosya in dosyalar:
        df = pd.read_json(dosya, lines=True, dtype={"id": str})
        df = df.set_index("id", verify_integrity=True)[alanlar].sort_index()
        tablolar.append(df)
    if any(not tablolar[0].equals(df) for df in tablolar[1:]):
        raise ValueError(f"{model}, {split}: farklı değerlendirme çıktıları var: {dosyalar}")
    return tablolar[0]

turler, degisimler = [], []
kayitlar = {}

for split in ["test-clean", "test-other"]:
    t = {m: tahminleri_oku(m, split) for m in modeller}
    for m in ["C", "D"]:
        if not t["B"][["reference", "N"]].equals(t[m][["reference", "N"]]):
            raise ValueError(f"{split}: {m} ile B'nin kayıtları veya referansları farklı.")

    kayitlar[split] = t
    hatalar = {}
    for m, df in t.items():
        s, d, i, n = df[["S", "D", "I", "N"]].sum()
        hatalar[m] = df[["S", "D", "I"]].sum(axis=1)
        turler.append([split, m, s, d, i, n, round(100 * (s+d+i) / n, 2)])

    for once, sonra in [("B", "C"), ("B", "D"), ("C", "D")]:
        fark = hatalar[sonra] - hatalar[once]
        degisimler.append([
            split, f"{once} → {sonra}",
            (fark < 0).sum(), (fark == 0).sum(), (fark > 0).sum(),
            -fark[fark < 0].sum(), fark[fark > 0].sum(), fark.sum()
        ])

display(pd.DataFrame(turler, columns=[
    "Küme", "Model", "İkame", "Silme", "Ekleme", "Referans kelime", "WER (%)"
]))
display(pd.DataFrame(degisimler, columns=[
    "Küme", "Karşılaştırma", "Hatası azalan kayıt", "Hata sayısı eşit kayıt",
    "Hatası artan kayıt", "Azalan hata toplamı", "Artan hata toplamı", "Net hata farkı"
]))

,Küme,Model,İkame,Silme,Ekleme,Referans kelime,WER (%)
0,test-clean,B,3010,413,1013,52576,8.44
1,test-clean,C,3199,488,993,52576,8.90
2,test-clean,D,3167,514,879,52576,8.67
3,test-other,B,5404,632,942,52343,13.33
4,test-other,C,5804,789,937,52343,14.39
5,test-other,D,5890,717,990,52343,14.51


,Küme,Karşılaştırma,Hatası azalan kayıt,Hata sayısı eşit kayıt,Hatası artan kayıt,Azalan hata toplamı,Artan hata toplamı,Net hata farkı
0,test-clean,B → C,91,2196,333,300,544,244
1,test-clean,B → D,100,2183,337,428,552,124
2,test-clean,C → D,47,2527,46,194,74,-120
3,test-other,B → C,173,2216,550,321,873,552
4,test-other,B → D,188,2171,580,310,929,619
5,test-other,C → D,76,2779,84,92,159,67


### 3.5.3 C-best ve D-best — Tam eşleşme ve %10 WER eşiği

Bir önceki hücrede eşleştirilen tahminler kullanılır. Baş ve sondaki
boşluklar ve harf büyüklüğü farkları göz ardı edilerek tam eşleşmeler
sayılır. Ayrıca kayıtların %10 WER eşiğinin hangi yönünde geçiş
yaptığı gösterilir. C’de %10 WER’i aşan kayıtlar seçilir ve iki
modelin toplam WER’i aynı kayıtlar üzerinde karşılaştırılır.

In [ ]:
import pandas as pd

eslesme_ozeti = []
esik_ozeti = []

for split in ["test-clean", "test-other"]:
    c = kayitlar[split]["C"]
    d = kayitlar[split]["D"]

    def tam_eslesme(df):
        ref = df["reference"].str.strip().str.casefold()
        tahmin = df["hypothesis"].str.strip().str.casefold()
        return tahmin.eq(ref)

    em_c = tam_eslesme(c)
    em_d = tam_eslesme(d)
    kazanilan = (~em_c & em_d).sum()
    kaybedilen = (em_c & ~em_d).sum()

    eslesme_ozeti.append({
        "Küme": split,
        "C tam eşleşme": em_c.sum(),
        "D tam eşleşme": em_d.sum(),
        "Yalnız D tam doğru": kazanilan,
        "Yalnız C tam doğru": kaybedilen,
        "Net kazanım (D−C)": kazanilan - kaybedilen,
    })

    hata_c = c[["S", "D", "I"]].sum(axis=1)
    hata_d = d[["S", "D", "I"]].sum(axis=1)
    n = c["N"]  # Her kaydın referans kelime sayısı

    # 10 × hata > N koşulu, WER > %10 ile eşdeğerdir.
    # Tam %10 olan kayıt, ödül fonksiyonunda alt aralıkta kalır.
    yuksek_c = 10 * hata_c > n
    yuksek_d = 10 * hata_d > n

    # Her iki modelin WER'ini aynı kayıtlar ve aynı payda ile hesapla.
    kelime_toplami = n[yuksek_c].sum()
    esik_ozeti.append({
        "Küme": split,
        "C'de %10 üstü kayıt": yuksek_c.sum(),
        "D'de %10 üstü kayıt": yuksek_d.sum(),
        "C > %10 → D ≤ %10": (yuksek_c & ~yuksek_d).sum(),
        "C ≤ %10 → D > %10": (~yuksek_c & yuksek_d).sum(),
        "C'de %10 üstü kayıtlarda C WER (%)":
            100 * hata_c[yuksek_c].sum() / kelime_toplami if kelime_toplami else None,
        "Aynı kayıtlarda D WER (%)":
            100 * hata_d[yuksek_c].sum() / kelime_toplami if kelime_toplami else None,
    })

display(pd.DataFrame(eslesme_ozeti))
display(pd.DataFrame(esik_ozeti).round(2))

,Küme,C tam eşleşme,D tam eşleşme,Yalnız D tam doğru,Yalnız C tam doğru,Net kazanım (D−C)
0,test-clean,1105,1104,11,12,-1
1,test-other,742,732,4,14,-10


,Küme,C'de %10 üstü kayıt,D'de %10 üstü kayıt,C > %10 → D ≤ %10,C ≤ %10 → D > %10,C'de %10 üstü kayıtlarda C WER (%),Aynı kayıtlarda D WER (%)
0,test-clean,681,673,17,9,27.04,25.78
1,test-other,1514,1515,12,13,24.44,24.59


### 3.5.4 D eğitimi — Aday gruplarının analizi

Eğitim günlüğünde aday metinleri kaydedilmiş gruplar incelenir.
Her grupta tam eşleşen, tek kelime hatası yapan ve iki veya daha
fazla hata yapan adaylar sayılır. Özellikle tam eşleşme bulunmayan,
ancak hem tek hatalı hem de daha çok hatalı aday içeren gruplar
belirlenir. Adım kayıtlarından ilk 50, son 50 ve tüm kayıtlı
adımlardaki aday düzeyi tam eşleşme oranı da hesaplanır.

In [ ]:
from pathlib import Path
import json
import runpy
import pandas as pd

root = Path("/content/drive/MyDrive/slam_asr")
run = root / "runs/D_qwen08_em40_wer60_casefold_s32_g16_train960"
errors = runpy.run_path(str(root / "05_compare_dev_other.py"))["errors"]

with (run / "metrics.jsonl").open() as f:
    logs = [json.loads(line) for line in f if line.strip()]

gruplar = []
for row in logs:
    if "candidates" not in row:
        continue

    ref = row["reference"]
    adaylar = row["candidates"]
    tam = sum(h.strip().casefold() == ref.strip().casefold() for h in adaylar)

    hata_sayilari = []
    for h in adaylar:
        counts = errors(ref, h)
        hata_sayilari.append(counts["S"] + counts["D"] + counts["I"])

    gruplar.append({
        "kayıt": row["example"],
        "aday": len(adaylar),
        "tam": tam,
        "bir_hata": hata_sayilari.count(1),
        "iki_ve_üzeri": sum(n >= 2 for n in hata_sayilari),
        "farklı_hata_sayısı": len(set(hata_sayilari)),
    })

g = pd.DataFrame(gruplar)
if g.empty:
    print("Bu günlükte aday metinleri bulunamadı.")
else:
    ara_odul = (g["tam"] == 0) & (g["bir_hata"] > 0) & (g["iki_ve_üzeri"] > 0)
    karma_em = (g["tam"] > 0) & (g["tam"] < g["aday"])

    print(f"Adayları kaydedilmiş grup: {len(g)}")
    for ad, maske in {
        "Hiç tam eşleşme olmayan": g["tam"] == 0,
        "Tam eşleşen ve eşleşmeyen adayları birlikte içeren": karma_em,
        "Tam eşleşme yok, hem 1 hem 2+ hatalı aday var": ara_odul,
    }.items():
        print(f"{ad}: {maske.sum()}/{len(g)} (%{100 * maske.mean():.1f})")

    display(g.loc[ara_odul].head(10))

adimlar = pd.DataFrame([r for r in logs if "em_count" in r and "candidate_count" in r])
if not adimlar.empty:
    adimlar = adimlar.drop_duplicates("step", keep="last").sort_values("step")
    for ad, kesit in [("İlk 50 adım", adimlar.head(50)),
                      ("Son 50 adım", adimlar.tail(50)),
                      ("Tüm kayıtlı adımlar", adimlar)]:
        oran = 100 * kesit["em_count"].sum() / kesit["candidate_count"].sum()
        print(f"{ad}: aday düzeyinde tam eşleşme %{oran:.2f}")

Adayları kaydedilmiş grup: 32
Hiç tam eşleşme olmayan: 20/32 (%62.5)
Tam eşleşen ve eşleşmeyen adayları birlikte içeren: 11/32 (%34.4)
Tam eşleşme yok, hem 1 hem 2+ hatalı aday var: 9/32 (%28.1)


,kayıt,aday,tam,bir_hata,iki_ve_üzeri,farklı_hata_sayısı
3,7150-118730-0022,16,0,9,7,3
4,868-131295-0004,16,0,3,13,5
7,2256-137628-0036,16,0,2,14,5
10,4018-107312-0027,16,0,1,15,6
22,1421-8522-0008,16,0,2,14,6
24,6135-41995-0038,16,0,1,15,6
28,1323-149236-0019,16,0,3,13,6
29,1165-135220-0042,16,0,4,12,5
31,6115-58433-0016,16,0,1,15,6


İlk 50 adım: aday düzeyinde tam eşleşme %9.95
Son 50 adım: aday düzeyinde tam eşleşme %11.24
Tüm kayıtlı adımlar: aday düzeyinde tam eşleşme %10.34


### 3.5.6 Tek hata bonusu — Eğitim öncesi yeniden puanlama

D koşusunun kayıtlı adayları, tek kelime hatasında 0,05, 0,10
veya 0,20 ek ödül verilerek yeniden puanlanır. Her grup için
GRPO avantajları yeniden hesaplanır. Tek hatalı adayların
avantajındaki değişim ve tam eşleşen adaylarla aralarındaki
avantaj farkı raporlanır. Bu hücre yeni model eğitimi başlatmaz.

In [ ]:
import numpy as np
import pandas as pd

bonuslar = [0.05, 0.10, 0.20]
sonuclar = []

def avantaj(puanlar):
    return (puanlar - puanlar.mean()) / (puanlar.std(ddof=0) + 1e-6)

for row in logs:
    if "candidates" not in row:
        continue

    ref = row["reference"]
    adaylar = row["candidates"]
    eski_puan = np.asarray(row["rewards"], dtype=float)
    eski_avantaj = avantaj(eski_puan)

    tam = np.array([
        hyp.strip().casefold() == ref.strip().casefold()
        for hyp in adaylar
    ])

    hata = []
    for hyp in adaylar:
        sayim = errors(ref, hyp)
        hata.append(sayim["S"] + sayim["D"] + sayim["I"])
    hata = np.asarray(hata)
    tek = hata == 1

    if not tek.any():
        continue

    if not tam.any() and (hata >= 2).any():
        grup = "Tam yok; tek ve çok hatalı aday var"
    elif tam.any():
        grup = "Tam ve tek hatalı aday var"
    else:
        continue

    for bonus in bonuslar:
        yeni_avantaj = avantaj(eski_puan + bonus * tek)
        sonuc = {
            "kayıt": row["example"],
            "grup": grup,
            "bonus": bonus,
            "tek_hata_avantaj_değişimi":
                (yeni_avantaj[tek] - eski_avantaj[tek]).mean(),
        }

        if tam.any():
            eski_fark = eski_avantaj[tam].mean() - eski_avantaj[tek].mean()
            yeni_fark = yeni_avantaj[tam].mean() - yeni_avantaj[tek].mean()
            sonuc["tam_tek_farkı_önce"] = eski_fark
            sonuc["tam_tek_farkı_sonra"] = yeni_fark
            sonuc["tam_tek_farkı_değişimi"] = yeni_fark - eski_fark

        sonuclar.append(sonuc)

detay = pd.DataFrame(sonuclar)
ozet = detay.groupby(["grup", "bonus"], sort=False).agg(
    grup_sayısı=("kayıt", "size"),
    ortalama_tek_hata_avantaj_değişimi=("tek_hata_avantaj_değişimi", "mean"),
    medyan_tek_hata_avantaj_değişimi=("tek_hata_avantaj_değişimi", "median"),
)
display(ozet.round(3))

karma = detay[detay["grup"] == "Tam ve tek hatalı aday var"]
if not karma.empty:
    display(karma.groupby("bonus")[[
        "tam_tek_farkı_önce", "tam_tek_farkı_sonra", "tam_tek_farkı_değişimi"
    ]].mean().round(3))
else:
    print("Aynı grupta hem tam eşleşen hem tek hatalı aday bulunamadı.")

grup_sayısı  \
grup                                bonus                
Tam yok; tek ve çok hatalı aday var 0.05             9   
                                    0.10             9   
                                    0.20             9   
Tam ve tek hatalı aday var          0.05            10   
                                    0.10            10   
                                    0.20            10   

                                           ortalama_tek_hata_avantaj_değişimi  \
grup                                bonus                                       
Tam yok; tek ve çok hatalı aday var 0.05                                0.606   
                                    0.10                                0.883   
                                    0.20                                1.127   
Tam ve tek hatalı aday var          0.05                                0.124   
                                    0.10                                0.251   
                                    0.20                                0.494   

                                           medyan_tek_hata_avantaj_değişimi  
grup                                bonus                                    
Tam yok; tek ve çok hatalı aday var 0.05                              0.583  
                                    0.10                              0.779  
                                    0.20                              1.133  
Tam ve tek hatalı aday var          0.05                              0.127  
                                    0.10                              0.269  
                                    0.20                              0.576

,tam_tek_farkı_önce,tam_tek_farkı_sonra,tam_tek_farkı_değişimi
bonus,,,
0.05,2.591,2.394,-0.197
0.10,2.591,2.153,-0.438
0.20,2.591,1.608,-0.982


## 3.6 Deney F-0,05 — Tek kelime hatasına ek ödülle GRPO

Eğitim B-best projektöründen başlar. D ödülüne, toplamda tam bir
kelime hatası yapan adaylar için 0,05 eklenir. Her adımda 32 ses
kaydı ve kayıt başına 16 aday kullanılır. Eğitim bütçesi 810 adımdır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 12_grpo_qwen_oneerror.py \
    --b-run B_qwen08_train960 \
    --run-name F_qwen08_em40_wer60_oneerror005_s32_g16_s810 \
    --steps 810 \
    --sounds-per-step 32 \
    --group-size 16 \
    --lr 1e-6 \
    --beta 0.04 \
    --clip 0.2 \
    --em-weight 0.4 \
    --wer-threshold 0.10 \
    --excess-penalty 2 \
    --em-ignore-case \
    --max-new-tokens 160 \
    --eval-every 200 \
    --save-every 10 \
    --seed 42

### 3.6.1 Deney F-0,05 — Doğrulama ve test değerlendirmesi

F-0,05 koşusunun 810. adımdaki son kontrol noktası (`last`),
tam `dev-other`, `test-clean` ve `test-other` kümelerinde değerlendirilir.
Transkriptler greedy decoding (`beam=1`) ile üretilir. WER değerleri
ve kayıt düzeyindeki tahminler `DF_oneerror005_full_eval` klasörüne yazılır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 10_evaluate_asr.py \
    --runs F_qwen08_em40_wer60_oneerror005_s32_g16_s810 \
    --checkpoints last \
    --splits dev-other test-clean test-other \
    --beams 1 --max-new-tokens 160 \
    --output DF_oneerror005_full_eval

/content/drive/MyDrive/slam_asr
Loading weights: 100% 479/479 [00:00<00:00, 28849.81it/s]
Loading weights: 100% 320/320 [00:00<00:00, 16393.81it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but mu

## 3.7 Deney B′ — Projektörde ReLU yerine Identity

B deneyinin model kimliği ve eğitim ayarları `config.json`
dosyasından alınır. Eğitim ve değerlendirme betiklerinin
Identity sürümleri oluşturulur. Projektördeki iki doğrusal
katman korunur ve aralarındaki ReLU, Identity ile değiştirilir.

B′ projektörü ayrı bir koşuda eğitilir. Bu karşılaştırma,
projektördeki doğrusal olmayan aktivasyonun etkisini inceler.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

import json
import subprocess
import sys
from pathlib import Path

base = json.loads(Path("runs/B_qwen08_train960/config.json").read_text())
run = "Bprime_qwen08_identity_train960"

def identity_copy(source, target):
    code = Path(source).read_text()
    if code.count("nn.ReLU()") != 1:
        raise ValueError(f"{source}: beklenen tek ReLU katmanı bulunamadı.")
    code = code.replace("nn.ReLU()", "nn.Identity()")
    compile(code, target, "exec")
    Path(target).write_text(code)

train_script = "08_qwen35_asr_identity.py"
eval_script = "10_evaluate_asr_identity.py"
identity_copy("08_qwen35_asr.py", train_script)
identity_copy("10_evaluate_asr.py", eval_script)

settings = [
    "batch_size", "gradient_accumulation_steps", "lr", "warmup_steps",
    "weight_decay", "max_steps", "save_every", "log_every", "eval_every",
    "dev_samples", "patience", "num_beams", "max_new_tokens", "seed",
]
cmd = [
    sys.executable, "-u", train_script,
    "--mode", "train", "--qwen", base["qwen"], "--run-name", run,
]
for key in settings:
    cmd.extend(["--" + key.replace("_", "-"), str(base[key])])

print("Başlatılan koşu:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

### 3.7.1 Deney B′ — Doğrulama ve test değerlendirmesi

`best` ve `last` projektör dosyalarının mevcut ve dolu olduğu
kontrol edilir. İki kontrol noktası `dev-other`, `test-clean`
ve `test-other` kümelerinde greedy decoding (`beam=1`) ile
değerlendirilir. Çıktılar `Bprime_identity_full_eval` klasörüne yazılır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

from pathlib import Path

run = "Bprime_qwen08_identity_train960"
for name in ("best_projector.pt", "projector.pt"):
    path = Path("runs") / run / name
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Checkpoint eksik veya boş: {path}")

!python -u 10_evaluate_asr_identity.py \
    --runs Bprime_qwen08_identity_train960 \
    --checkpoints best last \
    --splits dev-other test-clean test-other \
    --beams 1 \
    --max-new-tokens 160 \
    --output Bprime_identity_full_eval

## 3.8 Deney F-0,10 — Tek kelime hatasına ek ödülle GRPO

Eğitim B-best projektöründen başlar. D ödülüne, toplamda tam bir
kelime hatası yapan adaylar için 0,10 eklenir. Her adımda 32 ses
kaydı ve kayıt başına 16 aday kullanılır. Eğitim bütçesi 810 adımdır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr

!python -u 13_grpo_qwen_oneerror010.py \
    --b-run B_qwen08_train960 \
    --run-name F_qwen08_em40_wer60_oneerror010_s32_g16_s810 \
    --steps 810 \
    --sounds-per-step 32 \
    --group-size 16 \
    --lr 1e-6 \
    --beta 0.04 \
    --clip 0.2 \
    --em-weight 0.4 \
    --wer-threshold 0.10 \
    --excess-penalty 2 \
    --em-ignore-case \
    --max-new-tokens 160 \
    --eval-every 200 \
    --save-every 10 \
    --seed 42

### 3.8.1 Deney F-0,10 — Doğrulama ve test değerlendirmesi

F-0,10 koşusunun 810. adımdaki son kontrol noktası (`last`),
tam `dev-other`, `test-clean` ve `test-other` kümelerinde
greedy decoding (`beam=1`) ile değerlendirilir. WER değerleri
ve kayıt düzeyindeki tahminler `F_oneerror010_full_eval`
klasörüne yazılır.

In [ ]:
%cd /content/drive/MyDrive/slam_asr
!python -u 10_evaluate_asr.py \
    --runs F_qwen08_em40_wer60_oneerror010_s32_g16_s810 \
    --checkpoints last \
    --splits dev-other test-clean test-other \
    --beams 1 --max-new-tokens 160 \
    --output F_oneerror010_full_eval

### 3.8.2 D-810, F-0,05 ve F-0,10 — Kayıt düzeyinde hata analizi

Her iki test kümesinde tahminler kayıt kimliğine göre eşleştirilir
ve referansların aynı olduğu doğrulanır. D-810’a göre tek hata,
iki veya daha fazla hata ve tam eşleşme arasındaki seçili geçişler
sayılır. Ayrıca iyileşen ve kötüleşen kayıt sayıları, net hata
farkı ve en çok iyileşen beş kaydın giderilen hatalardaki payı
raporlanır.

In [ ]:
from pathlib import Path
import pandas as pd

runs = Path("/content/drive/MyDrive/slam_asr/runs")
modeller = {
    "D": "D_qwen08_em40_wer60_casefold_s32_g16_train960_step810",
    "F-0,05": "F_qwen08_em40_wer60_oneerror005_s32_g16_s810",
    "F-0,10": "F_qwen08_em40_wer60_oneerror010_s32_g16_s810",
}
alanlar = ["reference", "hypothesis", "S", "D", "I", "N"]

def tahminleri_oku(run, split, beklenen_kayit):
    yollar = sorted(runs.glob(f"*/{run}/last/{split}/predictions.jsonl"))
    if not yollar:
        raise FileNotFoundError(f"{run}/last/{split}/predictions.jsonl bulunamadı.")

    # Aynı koşu birden çok çıktı klasöründe varsa tam kopyaları karşılaştır.
    secilen = None
    for yol in yollar:
        df = pd.read_json(yol, lines=True, dtype={"id": str})
        if len(df) != beklenen_kayit:
            continue
        df = df.set_index("id", verify_integrity=True)[alanlar].sort_index()
        if secilen is not None and not secilen.equals(df):
            raise ValueError(f"Aynı koşu için farklı tam çıktılar bulundu: {yollar}")
        secilen = df
    if secilen is None:
        raise ValueError(f"{run}/{split}: {beklenen_kayit} kayıtlık tam çıktı bulunamadı.")
    return secilen

def durumlar(df):
    hata = df[["S", "D", "I"]].sum(axis=1)
    tam = df["reference"].str.strip().str.casefold().eq(
        df["hypothesis"].str.strip().str.casefold()
    )
    durum = pd.Series("2+", index=df.index)
    durum[hata.eq(1)] = "1"
    durum[hata.eq(0)] = "0 hata, metin farklı"
    durum[tam] = "Tam"
    return hata, durum

gecisler, katkilar = [], []
for split, adet in [("test-clean", 2620), ("test-other", 2939)]:
    d = tahminleri_oku(modeller["D"], split, adet)
    d_hata, once = durumlar(d)

    for model in ["F-0,05", "F-0,10"]:
        f = tahminleri_oku(modeller[model], split, adet)
        if not d[["reference", "N"]].equals(f[["reference", "N"]]):
            raise ValueError(f"{split}/{model}: kayıt kimlikleri veya referanslar farklı.")

        f_hata, sonra = durumlar(f)
        gecis = {"Küme": split, "D ile karşılaştırılan": model}
        for a, b in [("2+", "1"), ("1", "2+"), ("1", "Tam"), ("Tam", "1")]:
            gecis[f"{a} → {b}"] = (once.eq(a) & sonra.eq(b)).sum()
        gecis["Tam eşleşme farkı (F−D)"] = sonra.eq("Tam").sum() - once.eq("Tam").sum()
        gecisler.append(gecis)

        kazanc = d_hata - f_hata  # Pozitif değer: F bu kayıtta daha az hata yaptı.
        azalan = kazanc[kazanc > 0]
        artan = -kazanc[kazanc < 0]
        katkilar.append({
            "Küme": split, "Model": model,
            "İyileşen kayıt": len(azalan), "Kötüleşen kayıt": len(artan),
            "Giderilen hata": azalan.sum(), "Eklenen hata": artan.sum(),
            "Net giderilen hata": kazanc.sum(),
            "En iyi 5 kaydın giderilen hatalardaki payı (%)":
                round(100 * azalan.nlargest(5).sum() / azalan.sum(), 1)
                if len(azalan) else 0,
        })

display(pd.DataFrame(gecisler))
display(pd.DataFrame(katkilar))

### 3.8.3 F-0,05 — test-clean üzerinde en çok iyileşen beş kayıt

F-0,05’in D-810’a göre en fazla kelime hatası giderdiği beş kayıt
seçilir. Her kayıt için ikame, silme ve ekleme sayıları ile
referans ve iki modelin transkriptleri yan yana incelenir.
Bu örnekler, toplam WER kazanımına hangi düzeltmelerin katkı
verdiğini gösterir.

In [ ]:
import textwrap
import pandas as pd

split = "test-clean"
d = tahminleri_oku(modeller["D"], split, 2620)
f = tahminleri_oku(modeller["F-0,05"], split, 2620)

assert d[["reference", "N"]].equals(f[["reference", "N"]]), \
    "Kayıt kimlikleri veya referanslar eşleşmiyor."

tablo = pd.DataFrame(index=d.index)
tablo["Referans kelime"] = d["N"]

for kod, ad in [("S", "İkame"), ("D", "Silme"), ("I", "Ekleme")]:
    tablo[f"D: {ad}"] = d[kod]
    tablo[f"F: {ad}"] = f[kod]

tablo["D toplam hata"] = d[["S", "D", "I"]].sum(axis=1)
tablo["F toplam hata"] = f[["S", "D", "I"]].sum(axis=1)
tablo["Giderilen net hata"] = tablo["D toplam hata"] - tablo["F toplam hata"]

# Eşit kazanım varsa kayıt kimliği sırasını koru.
ilk5 = tablo.sort_values("Giderilen net hata", ascending=False, kind="stable").head(5)
display(ilk5)
print(f"Beş kayıtta giderilen net hata: {ilk5['Giderilen net hata'].sum()}")

for kayit, satir in ilk5.iterrows():
    print(f"\n{'=' * 72}")
    print(f"Kayıt {kayit} | Hata: {satir['D toplam hata']} → {satir['F toplam hata']}")
    for etiket, metin in [
        ("REFERANS", d.at[kayit, "reference"]),
        ("D-810", d.at[kayit, "hypothesis"]),
        ("F-0,05-810", f.at[kayit, "hypothesis"]),
    ]:
        print(f"\n{etiket}")
        print(textwrap.fill(metin, width=100) if metin else "[boş tahmin]")